In [1]:
# ── Cell 1: Imports & utility functions ─────────────────────────────────────
# Run once per session.

import json, os, math, shutil
import numpy as np
from pathlib import Path

import pybullet as p
import pybullet_data

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import imageio.v2 as imageio


# ── Camera specs ──────────────────────────────────────────────────────────────
CAMERAS = {
    "wrist": dict(w=640,  h=480,  fx=450.0,  fy=450.0,  cx=None, cy=None),
    "head":  dict(w=1920, h=1080, fx=1100.0, fy=1100.0, cx=None, cy=None),
}

def cam_spec(name):
    s = dict(CAMERAS[name])
    if s["cx"] is None: s["cx"] = s["w"] / 2.0
    if s["cy"] is None: s["cy"] = s["h"] / 2.0
    return s

def cam_K(name):
    s = cam_spec(name)
    return np.array([[s["fx"],0,s["cx"]],[0,s["fy"],s["cy"]],[0,0,1]], dtype=np.float64)

def cam_aspect(name):
    s = cam_spec(name)
    return float(s["w"]) / float(s["h"])


# ── Geometry helpers ──────────────────────────────────────────────────────────
def rot_x(deg):
    a = math.radians(deg); ca, sa = math.cos(a), math.sin(a)
    return np.array([[1,0,0],[0,ca,-sa],[0,sa,ca]], dtype=np.float64)

def rot_y(deg):
    a = math.radians(deg); ca, sa = math.cos(a), math.sin(a)
    return np.array([[ca,0,sa],[0,1,0],[-sa,0,ca]], dtype=np.float64)

def make_T(R, t_xyz):
    Tm = np.eye(4, dtype=np.float64)
    Tm[:3,:3] = R; Tm[:3,3] = np.asarray(t_xyz, dtype=np.float64)
    return Tm

def quat_xyzw_to_R(q):
    return np.array(p.getMatrixFromQuaternion(q), dtype=np.float64).reshape(3,3)

def cam2world_from_link(points, R_list, link_idx, offset_m, pitch_down_deg):
    T_world_link = make_T(R_list[link_idx], points[link_idx])
    T_link_cam   = make_T(rot_y(-pitch_down_deg), offset_m)
    return T_world_link @ T_link_cam

def make_head_cam2world(base_pos, height_m, pitch_down_deg):
    pos = np.array(base_pos, dtype=np.float64) + np.array([0, 0, height_m])
    return make_T(rot_y(-pitch_down_deg), pos)

def box_edges_from_T(T_world_obj, lwh):
    L, W, H = lwh; hx, hy, hz = L/2, W/2, H/2
    corners = np.array([[ hx, hy, hz],[ hx, hy,-hz],[ hx,-hy, hz],[ hx,-hy,-hz],
                        [-hx, hy, hz],[-hx, hy,-hz],[-hx,-hy, hz],[-hx,-hy,-hz]])
    corners_w = (T_world_obj[:3,:3] @ corners.T).T + T_world_obj[:3,3]
    edges = [(0,1),(0,2),(0,4),(3,1),(3,2),(3,7),(5,1),(5,4),(5,7),(6,2),(6,4),(6,7)]
    xs, ys, zs = [], [], []
    for a, b in edges:
        pa, pb = corners_w[a], corners_w[b]
        xs += [pa[0],pb[0],None]; ys += [pa[1],pb[1],None]; zs += [pa[2],pb[2],None]
    return np.array(xs,dtype=object), np.array(ys,dtype=object), np.array(zs,dtype=object)

def axes_lines_from_T(T, axis_len=0.10):
    o = T[:3,3]; R = T[:3,:3]
    return o, o+R[:,0]*axis_len, o+R[:,1]*axis_len, o+R[:,2]*axis_len


# ── PyBullet FK helpers ───────────────────────────────────────────────────────
def list_joints(robot_id):
    out = []
    for i in range(p.getNumJoints(robot_id)):
        info = p.getJointInfo(robot_id, i)
        out.append(dict(jid=int(info[0]), jname=info[1].decode(),
                        jtype=int(info[2]), parent=int(info[16]),
                        link_name=info[12].decode()))
    return out

def name_to_joint_index(robot_id):
    return {p.getJointInfo(robot_id,i)[1].decode(): i
            for i in range(p.getNumJoints(robot_id))}

def name_to_link_index(robot_id):
    d = {}
    for i in range(p.getNumJoints(robot_id)):
        info = p.getJointInfo(robot_id, i)
        d[info[12].decode()] = i   # link name
        d[info[1].decode()]  = i   # joint name
    return d

def get_links_world(robot_id):
    n = p.getNumJoints(robot_id)
    pts    = np.zeros((n, 3), dtype=np.float64)
    parent = np.full(n, -1,   dtype=np.int32)
    R_list = np.zeros((n, 3, 3), dtype=np.float64)
    for i in range(n):
        parent[i] = p.getJointInfo(robot_id, i)[16]
        st = p.getLinkState(robot_id, i, computeForwardKinematics=True)
        pts[i] = st[4]; R_list[i] = quat_xyzw_to_R(st[5])
    return pts, parent, R_list

def skeleton_lines(points, parent, mask=None):
    xs, ys, zs = [], [], []
    for i in range(len(points)):
        if mask is not None and not mask[i]: continue
        pidx = parent[i]
        if pidx >= 0:
            if mask is not None and not mask[pidx]: continue
            p1, p2 = points[pidx], points[i]
            xs+=[p1[0],p2[0],None]; ys+=[p1[1],p2[1],None]; zs+=[p1[2],p2[2],None]
    return np.array(xs,dtype=object), np.array(ys,dtype=object), np.array(zs,dtype=object)

def scene_ranges(points_list, cams_T_dict):
    all_pts = list(points_list)
    for Tm in cams_T_dict.values(): all_pts.append(Tm[:3,3][None,:])
    all_pts = np.vstack(all_pts)
    mins, maxs = all_pts.min(0), all_pts.max(0)
    center = (mins + maxs) / 2
    span   = max((maxs - mins).max(), 1e-6)
    return center, span / 2

def find_ee_link_index(robot_id, candidates):
    n2l = name_to_link_index(robot_id)
    for name in candidates:
        if name in n2l: return n2l[name], name
    raise RuntimeError("Cannot find EE link. Tried: " + ", ".join(candidates))

def set_arm_joints(robot_id, a, name2idx, joint_names):
    a = np.asarray(a, dtype=np.float64).reshape(-1)
    for k, jn in enumerate(joint_names):
        idx = name2idx.get(jn)
        if idx is None: raise RuntimeError(f"Joint not found in URDF: {jn}")
        p.resetJointState(robot_id, idx, float(a[k]))

def compute_frame(robot_id, f, name2idx, actions_6, joint_names):
    set_arm_joints(robot_id, actions_6[f], name2idx, joint_names)
    return get_links_world(robot_id)


# ── Trajectory loading ────────────────────────────────────────────────────────
def _read_jsonl(fp):
    rows = []
    with Path(fp).open('r', encoding='utf-8') as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise RuntimeError(f"JSON error in {fp} line {ln}: {e}") from e
    return rows

def _get_arm6_from_dict(d):
    arm = d.get("joint_positions") or d.get("qpos")
    if arm is None: raise KeyError("No 'joint_positions' or 'qpos' key found")
    arm = np.asarray(arm, dtype=np.float64).reshape(-1)
    if arm.shape[0] < 6: raise ValueError(f"Need >= 6 joint values, got {arm.shape[0]}")
    return arm[:6]

def build_trajectory(path):
    """Load a 6-DOF arm joint trajectory.

    Accepts:
      .jsonl  — one JSON dict per line with 'joint_positions' or 'qpos' key
      .npy    — array of shape (T, D) where D >= 6; first 6 columns are used

    Returns: np.ndarray of shape (T, 6), dtype float64.
    """
    path = Path(path)
    if path.suffix == ".npy":
        arr = np.load(path).astype(np.float64)
        if arr.ndim != 2 or arr.shape[1] < 6:
            raise ValueError(f".npy must be (T, D) with D>=6, got shape {arr.shape}")
        print(f"Loaded .npy  {arr.shape}  → using first 6 columns")
        return arr[:, :6]
    elif path.suffix == ".jsonl":
        rows = _read_jsonl(path)
        out  = np.zeros((len(rows), 6), dtype=np.float64)
        for t, d in enumerate(rows): out[t] = _get_arm6_from_dict(d)
        print(f"Loaded .jsonl {out.shape}")
        return out
    else:
        raise ValueError(f"Unsupported format '{path.suffix}'. Use .jsonl or .npy")

def pad_actions_edge(arr, target_len):
    arr = np.asarray(arr, dtype=np.float64)
    T0 = len(arr)
    if T0 == target_len: return arr
    if T0 >  target_len: return arr[:target_len]
    if T0 == 0:          return np.zeros((target_len, arr.shape[1]))
    return np.concatenate([arr, np.repeat(arr[-1:], target_len - T0, axis=0)])


print("✅ Cell 1: all imports and utility functions loaded.")

pybullet build time: Apr 13 2026 14:39:49


✅ Cell 1: all imports and utility functions loaded.


In [ ]:
# ── Cell 2: Configuration ────────────────────────────────────────────────────
# Edit paths here, then re-run Cell 3 to visualize.

# ── URDF ──────────────────────────────────────────────────────────────────────
URDF_PATH = "/liujinxin/code/tram/cosmos-predict2.5/outputs/ur5e/pybullet_ur5_robotiq-robotflow/urdf/ur5.urdf"

# ── Trajectories ──────────────────────────────────────────────────────────────
# Supported formats:
#   .jsonl  — one JSON dict per line with 'joint_positions' or 'qpos' key
#   .npy    — array of shape (T, D), D >= 6; first 6 columns used as joint angles
# EP1_PATH = Path("/liujinxin/code/lhc/wy/wms/lingbot-va/robochallenge/stack_color_blocks/data/episode_000000/states/states.jsonl")  # blue
EP1_PATH = Path("/liujinxin/code/lhc/wy/wms/lingbot-va/debug/stack_blocks_aligned/sample_002_idx281/pred_actions_denorm.npy")
EP2_PATH = Path("/liujinxin/code/lhc/wy/wms/lingbot-va/debug/stack_blocks_aligned/sample_002_idx281/gt_actions_denorm.npy")          # orange
# EP2_PATH = None  # set to None to show only trajectory 1

# ── Robot placement ───────────────────────────────────────────────────────────
BASE_POS = [0.0, 0.0, 0.0]
BASE_ORN = [0.0, 0.0, 0.0]         # Euler angles (rad)
ROBOT_COMPARE_OFFSET_Y_M = 0.0     # Y-axis visual offset for ep2 robot (not an error metric)

# ── Joint / link names ────────────────────────────────────────────────────────
UR5E_JOINTS = [
    "shoulder_pan_joint", "shoulder_lift_joint", "elbow_joint",
    "wrist_1_joint",      "wrist_2_joint",        "wrist_3_joint",
]
EE_CANDIDATE_NAMES = [
    "tool0", "ee_link", "tcp_link", "wrist_3_link", "wrist_3_joint", "ee_fixed_joint",
]

# ── Camera mounting ───────────────────────────────────────────────────────────
CAM_OFFSET_M        = np.array([0.06, 0.0, 0.04], dtype=np.float64)
CAM_PITCH_DOWN_DEG  = -25.0
HEAD_HEIGHT_M       = 0.60
HEAD_PITCH_DOWN_DEG = -30.0
CAM_BOX_LWH         = (0.06, 0.03, 0.03)

# ── Export ────────────────────────────────────────────────────────────────────
EXPORT_DIR        = "export_fk_ur5e_compare"
FPS               = 10
EXPORT_RENDER_CAM = "head"
SAVE_CAM_TXT      = True
CAM_TXT_DIRNAME   = "cameras"


In [5]:
# ── Cell 3: Initialize & Display ─────────────────────────────────────────────
# Re-run this cell whenever you change anything in Cell 2.

# ── Load trajectories ─────────────────────────────────────────────────────────
actions_6_ep1_raw = build_trajectory(EP1_PATH)
actions_6_ep2_raw = build_trajectory(EP2_PATH) if EP2_PATH is not None else actions_6_ep1_raw.copy()
ep2_visible = EP2_PATH is not None

T1, T2 = len(actions_6_ep1_raw), len(actions_6_ep2_raw)
T      = max(T1, T2)
actions_6_ep1 = pad_actions_edge(actions_6_ep1_raw, T)
actions_6_ep2 = pad_actions_edge(actions_6_ep2_raw, T)
print(f"ep1={T1} steps  ep2={T2} steps  display length T={T}")

# ── PyBullet: clean reconnect ─────────────────────────────────────────────────
try:    p.disconnect()
except: pass
p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())

base_pos_ep2 = [BASE_POS[0], BASE_POS[1] + ROBOT_COMPARE_OFFSET_Y_M, BASE_POS[2]]
robot_id_ep1 = p.loadURDF(URDF_PATH, basePosition=BASE_POS,
                          baseOrientation=p.getQuaternionFromEuler(BASE_ORN), useFixedBase=True)
robot_id_ep2 = p.loadURDF(URDF_PATH, basePosition=base_pos_ep2,
                          baseOrientation=p.getQuaternionFromEuler(BASE_ORN), useFixedBase=True)

name2idx_ep1 = name_to_joint_index(robot_id_ep1)
name2idx_ep2 = name_to_joint_index(robot_id_ep2)
ee_link_idx_ep1, ee_link_name_ep1 = find_ee_link_index(robot_id_ep1, EE_CANDIDATE_NAMES)
ee_link_idx_ep2, ee_link_name_ep2 = find_ee_link_index(robot_id_ep2, EE_CANDIDATE_NAMES)

print('\n=== URDF joints ===')
for j in list_joints(robot_id_ep1):
    print(f"  [{j['jid']:03d}] {j['jname']:<35s}  link={j['link_name']:<30s}  type={j['jtype']}")
print(f"EE link: {ee_link_name_ep1} (idx={ee_link_idx_ep1})")

T_world_head = make_head_cam2world(BASE_POS, HEAD_HEIGHT_M, HEAD_PITCH_DOWN_DEG)
mask_ep1 = np.ones(p.getNumJoints(robot_id_ep1), dtype=bool)
mask_ep2 = np.ones(p.getNumJoints(robot_id_ep2), dtype=bool)

# ── Helper: build camera transforms for a given frame ────────────────────────
def _cams_T_from_frame(pts, R_list):
    return {
        "wrist": cam2world_from_link(pts, R_list, ee_link_idx_ep1, CAM_OFFSET_M, CAM_PITCH_DOWN_DEG),
        "head":  T_world_head,
    }

def _title(f):
    base = f'UR5e FK  frame={f}  |  ep1=blue ({T1})'
    if ep2_visible:
        base += f'  |  ep2=orange ({T2})  Y_offset={ROBOT_COMPARE_OFFSET_Y_M:.3f}m'
    return base + f'  |  ee={ee_link_name_ep1}'

# ── Compute frame 0 ───────────────────────────────────────────────────────────
pts1, parent1, R1 = compute_frame(robot_id_ep1, 0, name2idx_ep1, actions_6_ep1, UR5E_JOINTS)
pts2, parent2, R2 = compute_frame(robot_id_ep2, 0, name2idx_ep2, actions_6_ep2, UR5E_JOINTS)
cams_T0 = _cams_T_from_frame(pts1, R1)
x1,y1,z1 = skeleton_lines(pts1, parent1, mask=mask_ep1)
x2,y2,z2 = skeleton_lines(pts2, parent2, mask=mask_ep2)

# ── Build Plotly figure ───────────────────────────────────────────────────────
fig = go.FigureWidget()

# traces 0-3: robot skeletons
fig.add_trace(go.Scatter3d(x=pts1[:,0], y=pts1[:,1], z=pts1[:,2],
    mode="markers", marker=dict(size=3, color="royalblue"), name="ep1_points"))
fig.add_trace(go.Scatter3d(x=x1, y=y1, z=z1,
    mode="lines", line=dict(width=5, color="royalblue"), name="ep1_bones"))
fig.add_trace(go.Scatter3d(x=pts2[:,0], y=pts2[:,1], z=pts2[:,2],
    mode="markers", marker=dict(size=3, color="orangered"), name="ep2_points", visible=ep2_visible))
fig.add_trace(go.Scatter3d(x=x2, y=y2, z=z2,
    mode="lines", line=dict(width=5, color="orangered"), name="ep2_bones", visible=ep2_visible))

# helper: add XYZ axis arrows for a transform
def _add_axes_traces(prefix, Tm, axis_len=0.15, visible=True):
    o, xe, ye, ze = axes_lines_from_T(Tm, axis_len)
    for end, col in [(xe,"red"),(ye,"green"),(ze,"blue")]:
        fig.add_trace(go.Scatter3d(
            x=[o[0],end[0]], y=[o[1],end[1]], z=[o[2],end[2]],
            mode="lines", line=dict(width=6, color=col),
            name=f"{prefix}_{col[0]}", visible=visible, showlegend=False))

def _add_box_trace(prefix, Tm, lwh, visible=True):
    x, y, z = box_edges_from_T(Tm, lwh)
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z,
        mode="lines", line=dict(width=4, color="purple"),
        name=f"{prefix}_box", visible=visible, showlegend=False))

trace_map = {
    "ep1_points": [0], "ep1_bones": [1],
    "ep2_points": [2], "ep2_bones": [3],
    "world_axes": [],
    "cam_axes":   {k: [] for k in CAMERAS},
    "cam_boxes":  {k: [] for k in CAMERAS},
}

# world origin axes (traces 4-6)
i0 = len(fig.data)
_add_axes_traces("world", np.eye(4), axis_len=0.20)
trace_map["world_axes"] = list(range(i0, i0+3))

# per-camera axes + box wireframe
for _cam in CAMERAS:
    i0 = len(fig.data)
    _add_axes_traces(_cam, cams_T0[_cam], axis_len=0.12)
    trace_map["cam_axes"][_cam] = list(range(i0, i0+3))
    i0 = len(fig.data)
    _add_box_trace(_cam, cams_T0[_cam], CAM_BOX_LWH)
    trace_map["cam_boxes"][_cam] = [i0]

center, half = scene_ranges([pts1, pts2], cams_T0)
fig.update_layout(
    title=_title(0), height=750, margin=dict(l=0,r=0,t=40,b=0),
    scene=dict(
        xaxis=dict(range=[center[0]-half, center[0]+half], title='X'),
        yaxis=dict(range=[center[1]-half, center[1]+half], title='Y'),
        zaxis=dict(range=[center[2]-half, center[2]+half], title='Z'),
        aspectmode='cube'),
    legend=dict(orientation='h'))

# ── Per-frame update (closes over robot state) ────────────────────────────────
def update_frame(f):
    f = int(f)
    p1, par1, r1 = compute_frame(robot_id_ep1, f, name2idx_ep1, actions_6_ep1, UR5E_JOINTS)
    p2, par2, r2 = compute_frame(robot_id_ep2, f, name2idx_ep2, actions_6_ep2, UR5E_JOINTS)
    cams_T_f = _cams_T_from_frame(p1, r1)
    ctr, hf  = scene_ranges([p1, p2], cams_T_f)
    x1,y1,z1 = skeleton_lines(p1, par1, mask=mask_ep1)
    x2,y2,z2 = skeleton_lines(p2, par2, mask=mask_ep2)
    with fig.batch_update():
        fig.data[0].x,fig.data[0].y,fig.data[0].z = p1[:,0],p1[:,1],p1[:,2]
        fig.data[1].x,fig.data[1].y,fig.data[1].z = x1,y1,z1
        fig.data[2].x,fig.data[2].y,fig.data[2].z = p2[:,0],p2[:,1],p2[:,2]
        fig.data[3].x,fig.data[3].y,fig.data[3].z = x2,y2,z2
        idx = 4 + 3  # skip world axes (3 traces)
        for _cam in CAMERAS:
            Tm = cams_T_f[_cam]
            o, xe, ye, ze = axes_lines_from_T(Tm, 0.12)
            for k, end in enumerate([xe, ye, ze]):
                fig.data[idx+k].x=[o[0],end[0]]; fig.data[idx+k].y=[o[1],end[1]]; fig.data[idx+k].z=[o[2],end[2]]
            idx += 3
            bx,by,bz = box_edges_from_T(Tm, CAM_BOX_LWH)
            fig.data[idx].x=bx; fig.data[idx].y=by; fig.data[idx].z=bz
            idx += 1
        fig.layout.title = _title(f)
        fig.layout.scene.xaxis.range=[ctr[0]-hf, ctr[0]+hf]
        fig.layout.scene.yaxis.range=[ctr[1]-hf, ctr[1]+hf]
        fig.layout.scene.zaxis.range=[ctr[2]-hf, ctr[2]+hf]
    apply_visibility()

# ── Matplotlib renderer (used by export) ─────────────────────────────────────
def render_frame_matplotlib(f, render_cam=None):
    if render_cam is None: render_cam = EXPORT_RENDER_CAM
    p1,par1,r1 = compute_frame(robot_id_ep1, f, name2idx_ep1, actions_6_ep1, UR5E_JOINTS)
    p2,par2,r2 = compute_frame(robot_id_ep2, f, name2idx_ep2, actions_6_ep2, UR5E_JOINTS)
    cams_T_f   = _cams_T_from_frame(p1, r1)
    aspect     = cam_aspect(render_cam)
    fig_m      = plt.figure(figsize=(6*aspect, 6))
    ax         = fig_m.add_subplot(111, projection='3d')
    ax.scatter(p1[:,0],p1[:,1],p1[:,2], s=10, c='royalblue')
    x1,y1,z1 = skeleton_lines(p1, par1, mask=mask_ep1)
    ax.plot(x1.astype(float),y1.astype(float),z1.astype(float), lw=2, c='royalblue')
    if ep2_visible:
        ax.scatter(p2[:,0],p2[:,1],p2[:,2], s=10, c='orangered')
        x2,y2,z2 = skeleton_lines(p2, par2, mask=mask_ep2)
        ax.plot(x2.astype(float),y2.astype(float),z2.astype(float), lw=2, c='orangered')
    for Tm in cams_T_f.values():
        o=Tm[:3,3]; RR=Tm[:3,:3]
        for col_i, col_name in enumerate(['r','g','b']):
            ax.quiver(o[0],o[1],o[2], RR[0,col_i],RR[1,col_i],RR[2,col_i], length=0.08, color=col_name)
    all_pts = np.vstack([p1, p2]+[T[:3,3][None,:] for T in cams_T_f.values()])
    mins,maxs = all_pts.min(0),all_pts.max(0)
    cen=(mins+maxs)/2; hf=max((maxs-mins).max(),1e-6)/2
    ax.set_xlim(cen[0]-hf,cen[0]+hf); ax.set_ylim(cen[1]-hf,cen[1]+hf); ax.set_zlim(cen[2]-hf,cen[2]+hf)
    ax.set_box_aspect([1,1,1]); ax.axis('off'); ax.set_title(_title(f))
    fig_m.canvas.draw()
    w, h = fig_m.canvas.get_width_height()
    img = np.frombuffer(fig_m.canvas.tostring_rgb(), dtype=np.uint8).reshape(h, w, 3)
    plt.close(fig_m)
    return img, cams_T_f

def _maybe_save_cam_txt(cams_T_f, frame_idx):
    if not SAVE_CAM_TXT: return
    cams_dir = os.path.join(EXPORT_DIR, CAM_TXT_DIRNAME)
    os.makedirs(cams_dir, exist_ok=True)
    for name in CAMERAS:
        np.savetxt(os.path.join(cams_dir, f"intrinsic_{name}.txt"), cam_K(name), fmt="%.9f")
    for name, Tm in cams_T_f.items():
        np.savetxt(os.path.join(cams_dir, f"extrinsic_{name}_f{frame_idx:06d}.txt"), Tm, fmt="%.9f")

def export_frames(export_type):
    os.makedirs(EXPORT_DIR, exist_ok=True)
    with out_log:
        clear_output()
        print(f"[export] type={export_type}  frames={T}  cam={EXPORT_RENDER_CAM}")
    if export_type == "gif":
        out_path = os.path.join(EXPORT_DIR, "fk_compare.gif")
        frames_buf = []
        for f in range(T):
            img, ct = render_frame_matplotlib(f)
            frames_buf.append(img); _maybe_save_cam_txt(ct, f)
            if f % 10 == 0:
                with out_log: print(f"  frame {f}/{T-1}")
        imageio.mimsave(out_path, frames_buf, duration=1.0/FPS)
        with out_log: print(f"[export] saved → {out_path}")
    elif export_type == "mp4":
        out_path = os.path.join(EXPORT_DIR, "fk_compare.mp4")
        if shutil.which("ffmpeg") is None:
            with out_log: print("[export][ERROR] ffmpeg not found"); return
        writer = imageio.get_writer(out_path, fps=FPS, codec='libx264',
                                    format='FFMPEG', ffmpeg_params=['-pix_fmt','yuv420p'])
        try:
            for f in range(T):
                img, ct = render_frame_matplotlib(f)
                writer.append_data(img); _maybe_save_cam_txt(ct, f)
                if f % 10 == 0:
                    with out_log: print(f"  frame {f}/{T-1}")
        finally: writer.close()
        with out_log: print(f"[export] saved → {out_path}")
    else:
        with out_log: print(f"[export] unknown type: {export_type}")

# ── Visibility helpers ────────────────────────────────────────────────────────
def _set_visible(indices, v):
    for idx in indices: fig.data[idx].visible = v

def apply_visibility():
    cams_on = set(cam_select.value)
    with fig.batch_update():
        _set_visible(trace_map["ep1_points"], cb_ep1_pts.value)
        _set_visible(trace_map["ep1_bones"],  cb_ep1_bones.value)
        _set_visible(trace_map["ep2_points"], ep2_visible and cb_ep2_pts.value)
        _set_visible(trace_map["ep2_bones"],  ep2_visible and cb_ep2_bones.value)
        _set_visible(trace_map["world_axes"], cb_world.value)
        for _cam in CAMERAS:
            on = _cam in cams_on
            _set_visible(trace_map["cam_axes"][_cam],  cb_cam_axes.value and on)
            _set_visible(trace_map["cam_boxes"][_cam], cb_cam_box.value  and on)

# ── Widgets ───────────────────────────────────────────────────────────────────
slider   = widgets.IntSlider(value=0, min=0, max=T-1, step=1,
                             description='frame', continuous_update=False)
play     = widgets.Play(interval=int(1000/FPS), value=0, min=0, max=T-1, step=1)
speed    = widgets.IntSlider(value=FPS, min=1, max=60,
                             description='fps', continuous_update=False)
cb_loop  = widgets.Checkbox(value=False, description='Loop')
widgets.jslink((play, 'value'), (slider, 'value'))

cb_ep1_pts   = widgets.Checkbox(value=True,        description='ep1 points')
cb_ep1_bones = widgets.Checkbox(value=True,        description='ep1 bones')
cb_ep2_pts   = widgets.Checkbox(value=ep2_visible, description='ep2 points')
cb_ep2_bones = widgets.Checkbox(value=ep2_visible, description='ep2 bones')
cb_world     = widgets.Checkbox(value=True, description='World axes')
cb_cam_axes  = widgets.Checkbox(value=True, description='Cam axes')
cb_cam_box   = widgets.Checkbox(value=True, description='Cam box')
cam_select   = widgets.SelectMultiple(options=list(CAMERAS), value=tuple(CAMERAS),
                                      description='Cams', rows=len(CAMERAS))
dd_export    = widgets.Dropdown(options=['none','gif','mp4'], value='none', description='Export')
btn_export   = widgets.Button(description='Export', button_style='info')
out_log      = widgets.Output()

def _on_speed(change): play.interval = int(1000 / max(1, int(change['new'])))
def _on_play(change):
    if cb_loop.value and int(change['new']) >= T - 1: play.value = 0
def _on_export(_):
    typ = dd_export.value
    with out_log: clear_output()
    if typ == 'none':
        with out_log: print('Select gif or mp4 first.')
        return
    export_frames(typ)

speed.observe(_on_speed,                           names='value')
play.observe(_on_play,                             names='value')
slider.observe(lambda c: update_frame(c['new']),   names='value')
for _w in [cb_ep1_pts, cb_ep1_bones, cb_ep2_pts, cb_ep2_bones,
           cb_world, cb_cam_axes, cb_cam_box, cam_select]:
    _w.observe(lambda c: apply_visibility(), names='value')
btn_export.on_click(_on_export)
apply_visibility()

# ── Display ───────────────────────────────────────────────────────────────────
with out_log:
    print(f"ep1 ({T1} steps) = blue  |  ep2 ({T2} steps) = {'orange' if ep2_visible else 'hidden (EP2_PATH=None)'}")
    print(f"total frames T={T}  |  ee link={ee_link_name_ep1}")
display(
    widgets.HBox([slider]),
    widgets.HBox([play, speed, cb_loop]),
    widgets.HBox([cb_ep1_pts, cb_ep1_bones, cb_ep2_pts, cb_ep2_bones]),
    widgets.HBox([cb_world, cb_cam_axes, cb_cam_box]),
    widgets.HBox([cam_select, dd_export, btn_export]),
    out_log,
    fig,
)


Loaded .npy  (192, 7)  → using first 6 columns
Loaded .npy  (192, 7)  → using first 6 columns
ep1=192 steps  ep2=192 steps  display length T=192
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
ee_link
=== URDF joints ===b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
ee_link
  [000] shoulder_pan_joint                   link=shoulder_link                   type=0
  [001] shoulder_lift_joint                  link=upper_arm_link                  type=0
  [002] elbow_joint                          link=forearm_link                    type=0
  [003] wrist_1_joint                        link=wrist_1_lin

Output()

FigureWidget({
    'data': [{'marker': {'color': 'royalblue', 'size': 3},
              'mode': 'markers',
              'name': 'ep1_points',
              'type': 'scatter3d',
              'uid': '13e8db57-363a-43df-abdb-5f84ad986a65',
              'visible': True,
              'x': {'bdata': 'AAAAAAAAAAAAAAAAeqeMvwAAAMBCEcK/AAAAwN/84L8AAACgVkvhvwAAAICISeS/AAAAYNeV5L8=',
                    'dtype': 'f8'},
              'y': {'bdata': 'AAAAAAAAAAAAAABg3EvBPwAAAAAQkFo/AAAAAIDUo78AAADgX4irPwAAAMALk6Y/AAAAYC3fpT8=',
                    'dtype': 'f8'},
              'z': {'bdata': 'AAAAwB/Ttj8AAAAAINO2PwAAAAAAYt8/AAAA4Egu3j8AAADgSC7ePwAAAMCI394/AAAAAPyj2T8=',
                    'dtype': 'f8'}},
             {'line': {'color': 'royalblue', 'width': 5},
              'mode': 'lines',
              'name': 'ep1_bones',
              'type': 'scatter3d',
              'uid': '4d684a84-d189-4511-8cf7-02cd4f2137a5',
              'visible': True,
              'x': array([np.float64(0.0), 